In [1]:
import pandas as pd
import re

# Read the data from the CSV file (using absolute path)
df = pd.read_csv('/home/sheikh/Projects/Thesis/data/medium.csv')
print("\nFirst 3 rows:\n", df.head(3))


First 3 rows:
    grmd                    md_name  \
0  1436   LIMNOCHORDIA L945 MEDIUM   
1  1479   SOLIDESULFOVIBRIO MEDIUM   
2  1481  Bold's Basal Medium (BBM)   

                                            tex_text  
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$$...  
1  \mono{KH$_2$PO$_4$}                           ...  
2  \mono{Agar}{20g}\\mono{ Distilled water}{980mL...  


In [2]:
def pass1_clean(tex_text):
    
    # 1. remove \hspace{...} and \hspace*{...}
    tex_text = re.sub(r'\\hspace\*?\{[^}]+\}', '', tex_text)
    
    # 2. remove \sfi and clean up extra spaces
    tex_text = tex_text.replace(r'\sfi', '')
    tex_text = re.sub(r'\{\s+', '{', tex_text)
    
    # 3. replace \Mix with Mix
    tex_text = tex_text.replace(r'\Mix', 'Mix')
    
    # 4. greek letters
    tex_text = re.sub(r'\$\\mu\$', 'μ', tex_text)
    tex_text = tex_text.replace(r'\mu', 'μ')
    tex_text = re.sub(r'\$\\cdot\$', '·', tex_text)
    tex_text = tex_text.replace(r'\cdot', '·')
    tex_text = tex_text.replace(r'\alpha', 'α')
    tex_text = tex_text.replace(r'\beta', 'β')
    tex_text = tex_text.replace(r'\HCl', 'HCl')
    tex_text = tex_text.replace(r'\ge', '≥')
    
    return tex_text

# reapply to all mediums
df["tex_text_clean"] = df["tex_text"].apply(pass1_clean)

# verify again
print("Remaining \\sfi   :", df["tex_text_clean"].str.contains(r'\\sfi').sum())
print("Remaining \\hspace:", df["tex_text_clean"].str.contains(r'\\hspace').sum())
print("Remaining \\Mix   :", df["tex_text_clean"].str.contains(r'\\Mix').sum())
print("Remaining \\mu    :", df["tex_text_clean"].str.contains(r'\\mu').sum())
print("Remaining \\cdot  :", df["tex_text_clean"].str.contains(r'\\cdot').sum())

Remaining \sfi   : 0
Remaining \hspace: 0
Remaining \Mix   : 0
Remaining \mu    : 0
Remaining \cdot  : 0


In [3]:
def fix_concatenated_amounts(tex_text):
    
    # fix double backslash + add newlines
    tex_text = tex_text.replace('\\\\mono', '\n\\mono')
    tex_text = tex_text.replace('\\\\chu', '\n\\chu')
    
    # split concatenated {amountunit} → {amount} {unit}
    tex_text = re.sub(
        r'\{([0-9.]+)([a-zA-Z]+)\}',
        lambda m: ' {' + m.group(1) + '} {' + m.group(2).lower() + '}',
        tex_text
    )
    
    # fix missing space between name and amount: }{  →  } {
    tex_text = re.sub(r'\}(\s*)\{([0-9])', r'} {\2', tex_text)
    
    return tex_text

test_1481 = df[df["grmd"] == 1481]["tex_text_clean"].values[0]
fixed = fix_concatenated_amounts(test_1481)
print(fixed)

\mono{Agar} {20} {g}
\mono{Distilled water} {980} {ml}
\chu{Sigma-Aldrich Bold Modified Basal Freshwater Nutrient Solution, 50x 20mL
 Add 50x BBM after autoclave}
     



In [4]:
# apply to all mediums
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_concatenated_amounts)

# verify no concatenated amounts remain
concat_remaining = df[df["tex_text_clean"].str.contains(r'\{[0-9.]+[a-zA-Z]+\}')]
print(f"Concatenated remaining: {len(concat_remaining)}")

# verify no double backslash remains
double_bs = df[df["tex_text_clean"].str.contains(r'\\\\mono|\\\\chu')]
print(f"Double backslash remaining: {len(double_bs)}")

Concatenated remaining: 0
Double backslash remaining: 0


In [5]:
def pass3_units(tex_text):
    """Normalize unit variations"""
    
    # HTML entities
    tex_text = tex_text.replace('&mu;', 'μ')
    tex_text = tex_text.replace('&#956;', 'μ')
    
    # normalize case
    tex_text = re.sub(r'\{mL\}', '{ml}', tex_text)      # mL → ml
    
    # corrupted units
    tex_text = re.sub(r'\{vg\}', '{g}', tex_text)        # vg → g (corrupted })
    tex_text = re.sub(r'\{m\}', '{ml}', tex_text)        # m → ml (truncated)
    
    # LaTeX encoded units (after pass1 these should already be unicode)
    tex_text = tex_text.replace('$μ$g', 'μg')
    tex_text = tex_text.replace('$μ$l', 'μl')
    
    return tex_text

# test on a few known cases
df["tex_text_clean"] = df["tex_text_clean"].apply(pass3_units)

# verify
print("Remaining mL  :", df["tex_text_clean"].str.contains(r'\{mL\}').sum())
print("Remaining vg  :", df["tex_text_clean"].str.contains(r'\{vg\}').sum())
print("Remaining &mu;:", df["tex_text_clean"].str.contains(r'&mu;').sum())
print("Remaining &#956;:", df["tex_text_clean"].str.contains(r'&#956;').sum())

Remaining mL  : 0
Remaining vg  : 0
Remaining &mu;: 0
Remaining &#956;: 0


In [6]:
# extract all units from cleaned tex_text
unit_pattern = r'\\mono\{[^}]+\}\s*\{[^}]+\}\s*\{([^}]*)\}'
all_units_clean = []
for tex in df["tex_text_clean"]:
    matches = re.findall(unit_pattern, tex)
    all_units_clean.extend(matches)

from collections import Counter
unit_counts = Counter(u.strip() for u in all_units_clean)

print(f"Total \\mono lines with units: {len(all_units_clean)}")
print(f"Unique units found: {len(unit_counts)}")
print("\nAll units and their counts:")
for unit, count in unit_counts.most_common():
    print(f"  {repr(unit):30} → {count}")

Total \mono lines with units: 12742
Unique units found: 8

All units and their counts:
  'g'                            → 7229
  'ml'                           → 3038
  'mg'                           → 1530
  'L'                            → 883
  'μg'                           → 38
  'mM'                           → 12
  'μl'                           → 11
  'mg\n\\mono{Na$_2$MoO$_4$·2H$_2$O' → 1


In [8]:
def fix_missing_closing_brace(tex_text):
    """Fix \\mono lines with missing closing } on unit"""
    
    # pattern: {unit with no closing brace at end of line
    # \mono{name} {amount}{unit\n  ← missing }
    tex_text = re.sub(
        r'\{([a-zA-Zμ]+)\n',           # {unit followed by newline (no })
        r'{\1}\n',                      # add closing }
        tex_text
    )
    return tex_text

# test on medium 1103
test_1103 = df[df["grmd"] == 1103]["tex_text_clean"].values[0]
fixed = fix_missing_closing_brace(test_1103)

# print the relevant lines
for line in fixed.splitlines():
    if 'NiCl' in line or 'MoO' in line:
        print(repr(line.strip()))

'\\mono{NiCl$_2$·6H$_2$O} {24.0}{mg}'
'\\mono{Na$_2$MoO$_4$·2H$_2$O} {36.0}{mg}'


In [9]:
# apply fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_missing_closing_brace)

# recheck units
all_units_clean = []
for tex in df["tex_text_clean"]:
    matches = re.findall(unit_pattern, tex)
    all_units_clean.extend(matches)

unit_counts = Counter(u.strip() for u in all_units_clean)

print(f"Unique units found: {len(unit_counts)}")
print("\nAll units and their counts:")
for unit, count in unit_counts.most_common():
    print(f"  {repr(unit):30} → {count}")

Unique units found: 7

All units and their counts:
  'g'                            → 7229
  'ml'                           → 3038
  'mg'                           → 1532
  'L'                            → 883
  'μg'                           → 38
  'mM'                           → 12
  'μl'                           → 11


In [10]:
# first let's see how many remain
v_corrupted = df[df["tex_text_clean"].str.contains(r'\\mono\{[^}]*v\s*\{')]
print(f"Mediums with v instead of }}: {len(v_corrupted)}")

Mediums with v instead of }: 3


In [11]:
for idx, row in v_corrupted.iterrows():
    for line in row["tex_text_clean"].splitlines():
        if re.search(r'\\mono\{[^}]*v\s*\{', line):
            print(f"{row['grmd']} → {repr(line.strip())}")

1446 → '\\mono{Glucosev {5.0} {g}'
1430 → '\\mono{Distilled waterv                                         {1.0}{L}'
1347 → '\\mono{MnSO$_4$·xH$_2$Ov                                 {0.5}{g}'


In [12]:
def fix_corrupted_v(tex_text):
    """Fix v instead of } at end of component name in \\mono lines"""
    
    # pattern: \mono{...v followed by whitespace and {amount}
    tex_text = re.sub(
        r'(\\mono\{[^}]*)v(\s*\{)',  # v before whitespace + {
        r'\1}\2',                     # replace v with }
        tex_text
    )
    return tex_text

# test on the 3 mediums
for grmd in [1446, 1430, 1347]:
    test = df[df["grmd"] == grmd]["tex_text_clean"].values[0]
    fixed = fix_corrupted_v(test)
    for line in fixed.splitlines():
        if 'Glucose' in line or 'Distilled water' in line or 'MnSO' in line:
            print(f"{grmd} → {repr(line.strip())}")

1446 → '\\mono{Glucose} {5.0} {g}'
1446 → '\\mono{Distilled water} {1.0}{L}'
1430 → '\\mono{Distilled water}                                         {1.0}{L}'
1347 → '\\mono{Distilled water} {1.0}{L}'
1347 → '\\mono{MnSO$_4$·xH$_2$O}                                 {0.5}{g}'
1347 → '\\mono{Distilled water} {1.0}{L}'
1347 → '\\mono{Distilled water} {1.0}{L}'


In [13]:
# apply fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_corrupted_v)

# verify no v corruptions remain
v_remaining = df[df["tex_text_clean"].str.contains(r'\\mono\{[^}]*v\s*\{')]
print(f"Corrupted v remaining: {len(v_remaining)}")

Corrupted v remaining: 0


In [14]:
# how many \mono lines have missing } on component name?
missing_brace = df[df["tex_text_clean"].str.contains(
    r'\\mono\{[^}]+\n'  # \mono{ with no closing } before newline
)]
print(f"Mediums with missing }} on component name: {len(missing_brace)}")

for idx, row in missing_brace.head(5).iterrows():
    for line in row["tex_text_clean"].splitlines():
        if re.search(r'\\mono\{[^}]+$', line):
            print(f"{row['grmd']} → {repr(line.strip())}")

Mediums with missing } on component name: 1
1093 → '\\mono{MJ(-N) synthetic seawater'


In [15]:
test_1093 = df[df["grmd"] == 1093]["tex_text_clean"].values[0]
print(test_1093[:500])

\mono{MJ(-N) synthetic seawater
(see Medium No. [268])} {1.0}{L}
\mono{NH$_4$Cl} {0.25}{g}
\mono{KNO$_3$} {0.25}{g}
\mono{Distilled water} {1.0}{L}

\chu{Mix components thoroughly and autoclave. After cooling, add the following solutions (filter--sterilized):}

\mono{8\% NaHCO$_3$ solution} {10.0}{ml}
\mono{10\% Na$_2$S$_2$O$_3$·5H$_2$O solution} {15.0}{ml}
\mono{Trace vitamins (see Medium No. [197])} {1.0}{ml}

\chu{Aseptically distribute the medium into culture vessels (e.g., 20 ml in 120 ml s


In [16]:
def fix_multiline_mono_name(tex_text):
    """Fix \\mono component names that span multiple lines"""
    
    # pattern: \mono{... with no closing } at end of line
    # followed by continuation on next line
    tex_text = re.sub(
        r'(\\mono\{[^}]*)\n([^\\{]*\})',  # \mono{name\ncontinuation}
        r'\1 \2',                          # join with space
        tex_text
    )
    return tex_text

# test on 1093
test_1093 = df[df["grmd"] == 1093]["tex_text_clean"].values[0]
fixed = fix_multiline_mono_name(test_1093)

# print the fixed line
for line in fixed.splitlines():
    if 'seawater' in line:
        print(repr(line.strip()))

'\\mono{MJ(-N) synthetic seawater (see Medium No. [268])} {1.0}{L}'


In [17]:
# apply fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_multiline_mono_name)

# verify
missing_remaining = df[df["tex_text_clean"].str.contains(r'\\mono\{[^}]+\n')]
print(f"Multiline mono names remaining: {len(missing_remaining)}")

Multiline mono names remaining: 0


In [18]:
# how many have leading space in component name?
leading_space = df[df["tex_text_clean"].str.contains(r'\\mono\{\s+')]
print(f"Mediums with leading space in name: {len(leading_space)}")

for idx, row in leading_space.iterrows():
    for line in row["tex_text_clean"].splitlines():
        if re.search(r'\\mono\{\s+', line):
            print(f"{row['grmd']} → {repr(line.strip()[:60])}")

Mediums with leading space in name: 0


In [19]:
# find the 10 plain reference mediums
plain_ref_pattern = r'(?i)^(use|prepare)\s+(medium|cm|oatmeal|jcm|blood|aquifex)'
plain_refs = df[df["tex_text_clean"].str.strip().str.match(plain_ref_pattern)]
print(f"Plain references found: {len(plain_refs)}")
for idx, row in plain_refs.iterrows():
    first_line = row["tex_text_clean"].strip().splitlines()[0]
    print(f"{row['grmd']} → {first_line[:80]}")

Plain references found: 10
1281 → Use Medium No. [770], replacing betaine solution in Solution B with 1.0 M glucos
1233 → Prepare CM+YE medium (see Medium No. [59]). After autoclaving, adjust pH to 9.0 
1117 → Use Medium No. [1116] with 60.0 g/L NaCl (final).
818 → Use JCM Medium No. [791] with 5.0 g/L MgSO$_4$·7H$_2$O instead of CaCl$_2$·2H$_2
746 → Use Medium No. [598] without CaCO$_3$.
625 → Use Medium No. [624] supplemented with
627 → Use Medium No. [552] supplemented with 2\% (final) NaCl.
689 → Prepare Oatmeal agar (ISP--3) (see Medium No. [50]) and adjust pH to 6.0.
637 → Use Medium No. [636] without methanol.  Replace the
619 → Use Medium No. [618] with 16.0 g/L (final) NaCl.


In [20]:
def fix_plain_references(tex_text):
    """Wrap plain reference text missing \\chu{} wrapper"""
    
    # check if tex_text starts with plain reference (no tag)
    stripped = tex_text.strip()
    if re.match(r'(?i)^(use|prepare)\s+(medium|cm|oatmeal|jcm|blood|aquifex)', stripped):
        # wrap entire text in \chu{}
        tex_text = r'\chu{' + stripped + '}'
    
    return tex_text

# test on a few
for grmd in [1281, 1117, 746]:
    test = df[df["grmd"] == grmd]["tex_text_clean"].values[0]
    fixed = fix_plain_references(test)
    print(f"\n{grmd} BEFORE: {test[:80]}")
    print(f"{grmd} AFTER : {fixed[:80]}")


1281 BEFORE: Use Medium No. [770], replacing betaine solution in Solution B with 1.0 M glucos
1281 AFTER : \chu{Use Medium No. [770], replacing betaine solution in Solution B with 1.0 M g

1117 BEFORE: Use Medium No. [1116] with 60.0 g/L NaCl (final).
     

1117 AFTER : \chu{Use Medium No. [1116] with 60.0 g/L NaCl (final).}

746 BEFORE: Use Medium No. [598] without CaCO$_3$.
     

746 AFTER : \chu{Use Medium No. [598] without CaCO$_3$.}


In [21]:
# apply fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_plain_references)

# verify no plain references remain
plain_remaining = df[df["tex_text_clean"].str.strip().str.match(plain_ref_pattern)]
print(f"Plain references remaining: {len(plain_remaining)}")

# verify all 10 now start with \chu
for grmd in [1281, 1233, 1117, 818, 746, 625, 627, 689, 637, 619]:
    first_line = df[df["grmd"] == grmd]["tex_text_clean"].values[0].strip()[:60]
    print(f"{grmd} → {first_line}")

Plain references remaining: 0
1281 → \chu{Use Medium No. [770], replacing betaine solution in Sol
1233 → \chu{Prepare CM+YE medium (see Medium No. [59]). After autoc
1117 → \chu{Use Medium No. [1116] with 60.0 g/L NaCl (final).}
818 → \chu{Use JCM Medium No. [791] with 5.0 g/L MgSO$_4$·7H$_2$O 
746 → \chu{Use Medium No. [598] without CaCO$_3$.}
625 → \chu{Use Medium No. [624] supplemented with
3.0 g/L L--proli
627 → \chu{Use Medium No. [552] supplemented with 2\% (final) NaCl
689 → \chu{Prepare Oatmeal agar (ISP--3) (see Medium No. [50]) and
637 → \chu{Use Medium No. [636] without methanol.  Replace the
gas
619 → \chu{Use Medium No. [618] with 16.0 g/L (final) NaCl.}


In [22]:
# how many have double {{ remaining?
double_curly = df[df["tex_text_clean"].str.contains(r'\{\{')]
print(f"Mediums with double {{: {len(double_curly)}")

for idx, row in double_curly.head(5).iterrows():
    for line in row["tex_text_clean"].splitlines():
        if '{{' in line:
            print(f"{row['grmd']} → {repr(line.strip()[:80])}")

Mediums with double {: 72
1365 → '\\chu{{Selenite-tungstate solution:}'
1435 → '\\chu{{Trace element solution:}'
1429 → '\\mono{{N}-Acetyl-D-glucosamine} {1.0}{g}'
1419 → '\\chu{{Modified A5 solution:}'
1398 → '\\chu{{Neutralized sulfide solution:}'
1398 → '\\chu{{Vitamin solution CA:}'
1398 → '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}'


In [23]:
def fix_double_curly(tex_text):
    """Fix double {{ left behind after \\sfi removal"""
    
    # replace {{ with { but not inside math mode $...$
    tex_text = re.sub(r'\{\{', '{', tex_text)
    
    return tex_text

# test
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_double_curly)

# verify
double_remaining = df[df["tex_text_clean"].str.contains(r'\{\{')]
print(f"Double {{ remaining: {len(double_remaining)}")

Double { remaining: 0


In [24]:
# check each case
print("Extra text after name:", 
    df[df["tex_text_clean"].str.contains(r'\\mono\{[^}]+\}\s*\(')].shape[0])

print("Missing } on amount:", 
    df[df["tex_text_clean"].str.contains(r'\{[0-9.]+\s+\{')].shape[0])

print("HTML in md_name:", 
    df[df["md_name"].str.contains(r'&#\d+;', na=False)].shape[0])

Extra text after name: 1
Missing } on amount: 1
HTML in md_name: 1


In [25]:
# see the extra text after name
extra = df[df["tex_text_clean"].str.contains(r'\\mono\{[^}]+\}\s*\(')]
for idx, row in extra.iterrows():
    for line in row["tex_text_clean"].splitlines():
        if re.search(r'\\mono\{[^}]+\}\s*\(', line):
            print(f"Extra text → {row['grmd']}: {repr(line.strip())}")

# see missing } on amount
missing = df[df["tex_text_clean"].str.contains(r'\{[0-9.]+\s+\{')]
for idx, row in missing.iterrows():
    for line in row["tex_text_clean"].splitlines():
        if re.search(r'\{[0-9.]+\s+\{', line):
            print(f"Missing }} → {row['grmd']}: {repr(line.strip())}")

# see HTML in md_name
html = df[df["md_name"].str.contains(r'&#\d+;', na=False)]
for idx, row in html.iterrows():
    print(f"HTML name → {row['grmd']}: {repr(row['md_name'])}")

Extra text → 1353: '\\mono{Trace vitamins} (see Medium No. [197])} {1.0 {L}'
Missing } → 1353: '\\mono{Trace vitamins} (see Medium No. [197])} {1.0 {L}'
HTML name → 1087: 'POREMEDIA B-CYE&#945; AGAR MEDIUM'


In [26]:
# see full context of 1353
test_1353 = df[df["grmd"] == 1353]["tex_text_clean"].values[0]
print(test_1353)

\mono{NaCl} {3.0}{g}
\mono{KCl} {0.15}{g}
\mono{Na$_2$SO$_4$} {0.3}{g}
\mono{MgSO$_4$·7H$_2$O} {0.123}{g}
\mono{CaCl$_2$·2H$_2$O} {14.5}{mg}
\mono{NH$_4$Cl} {21.4}{mg}
\mono{HEPES} {0.477}{g}
\mono{Mineral solution A (see below)} {5.0}{ml}
\mono{Distilled water} {1.0}{L}
\chu{Mix components thoroughly and adjust pH to 7.5 with NaOH.  Distribute the medium into culture vessels (e.g., 5.0 ml in 25 ml serum bottles/Balch tubes) under a stream of N$_2$, seal with butyl rubber stoppers and autoclave.  After cooling, aseptically add per liter the following solutions (autoclaved or filter-sterilized):} 
\mono{Trace vitamins solution* (see below)} {10.0}{ml}
\mono{4% Glycogen solution} {10.0}{ml}
\mono{Mineral solution B* (see below)} {2.0}{ml}
\chu{Prior to inoculation, add filter-sterilized O$_2$ gas or air, to make a final concentration of 10% O$_2$ in the gas phase.}

\chu{Mineral solution A:}
\mono{EDTA} {2.5}{g}
\mono{KOH} {2.2}{g}
\mono{FeSO$_4$·7H$_2$O} {1.1}{g}
\mono{MnCl$_2$·4H$_2$O}

In [27]:
# find the exact problematic line
for line in test_1353.splitlines():
    if 'Trace vitamins' in line:
        print(repr(line.strip()))

'\\mono{Trace vitamins solution* (see below)} {10.0}{ml}'
'\\chu{Trace vitamins solution:}'
'\\mono{Trace vitamins} (see Medium No. [197])} {1.0 {L}'


In [28]:
def fix_1353(tex_text):
    """Fix malformed Trace vitamins line in medium 1353"""
    tex_text = tex_text.replace(
        r'\mono{Trace vitamins} (see Medium No. [197])} {1.0 {L}',
        r'\mono{Trace vitamins (see Medium No. [197])} {1.0} {L}'
    )
    return tex_text

# apply only to 1353
df.loc[df["grmd"] == 1353, "tex_text_clean"] = fix_1353(
    df[df["grmd"] == 1353]["tex_text_clean"].values[0]
)

# verify
test = df[df["grmd"] == 1353]["tex_text_clean"].values[0]
for line in test.splitlines():
    if 'Trace vitamins' in line:
        print(repr(line.strip()))

'\\mono{Trace vitamins solution* (see below)} {10.0}{ml}'
'\\chu{Trace vitamins solution:}'
'\\mono{Trace vitamins (see Medium No. [197])} {1.0} {L}'


In [29]:
# fix &#945; → α in md_name
df["md_name"] = df["md_name"].str.replace('&#945;', 'α', regex=False)
df["md_name"] = df["md_name"].str.replace('&#956;', 'μ', regex=False)

# verify
print(df[df["grmd"] == 1087]["md_name"].values[0])

POREMEDIA B-CYEα AGAR MEDIUM


In [30]:
print("=== FINAL CLEANING VERIFICATION ===\n")

# tags
print("Remaining \\sfi    :", df["tex_text_clean"].str.contains(r'\\sfi').sum())
print("Remaining \\hspace :", df["tex_text_clean"].str.contains(r'\\hspace').sum())
print("Remaining \\Mix    :", df["tex_text_clean"].str.contains(r'\\Mix').sum())
print("Remaining \\mu     :", df["tex_text_clean"].str.contains(r'\\mu').sum())
print("Remaining \\cdot   :", df["tex_text_clean"].str.contains(r'\\cdot').sum())

# braces
print("\nRemaining double {{ :", df["tex_text_clean"].str.contains(r'\{\{').sum())
print("Remaining v instead of }}:", df["tex_text_clean"].str.contains(r'\\mono\{[^}]*v\s*\{').sum())
print("Remaining missing }} on unit:", df["tex_text_clean"].str.contains(r'\{[0-9.]+\s+\{').sum())

# units
print("\nRemaining mL      :", df["tex_text_clean"].str.contains(r'\{mL\}').sum())
print("Remaining vg      :", df["tex_text_clean"].str.contains(r'\{vg\}').sum())
print("Remaining &mu;    :", df["tex_text_clean"].str.contains(r'&mu;').sum())

# plain references
print("\nRemaining plain refs:", df["tex_text_clean"].str.strip().str.match(plain_ref_pattern).sum())

# HTML in md_name
print("\nHTML in md_name   :", df["md_name"].str.contains(r'&#\d+;', na=False).sum())

=== FINAL CLEANING VERIFICATION ===

Remaining \sfi    : 0
Remaining \hspace : 0
Remaining \Mix    : 0
Remaining \mu     : 0
Remaining \cdot   : 0

Remaining double {{ : 0
Remaining v instead of }}: 0
Remaining missing }} on unit: 0

Remaining mL      : 0
Remaining vg      : 0
Remaining &mu;    : 0

Remaining plain refs: 0

HTML in md_name   : 0


In [31]:
# 1. how many \mono lines still missing amount/unit after all cleaning?
unit_pattern = r'\\mono\{[^}]+\}\s*\{[^}]+\}\s*\{[^}]*\}'
no_amount = []
for tex in df["tex_text_clean"]:
    lines = tex.splitlines()
    for line in lines:
        if '\\mono' in line and not re.search(unit_pattern, line):
            no_amount.append(line.strip())

print(f"\\mono lines still missing amount/unit: {len(no_amount)}")
print("\nExamples:")
for line in no_amount[:10]:
    print(repr(line[:80]))

\mono lines still missing amount/unit: 183

Examples:
'\\mono{N}-Acetyl-D-glucosamine} {1.0}{g}'
'\\mono{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{MgSO$_4$·7H$_2$O                                  {0.02}{g}'
'\\mono{1 M MgSO$_4$}solution} {1.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{MnCl$_2$·4H$_2$O                                  {1.81}{g}'
'\\mono{myo}-Inositol} {5.0}{mg}'
'\\mono{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {100.0}{mg}'


In [32]:
# how many of each pattern?
pattern1 = [l for l in no_amount if re.search(r'\\mono\{[A-Za-z]\}', l)]
pattern2 = [l for l in no_amount if re.search(r'\\mono\{[^}]+\s+\{', l)]
pattern3 = [l for l in no_amount if re.search(r'\\mono\{[^}]+\}[a-z]+\}', l)]

print(f"Pattern 1 (early closing }}) : {len(pattern1)}")
print(f"Pattern 2 (missing closing }}) : {len(pattern2)}")
print(f"Pattern 3 (extra text after name) : {len(pattern3)}")

Pattern 1 (early closing }) : 63
Pattern 2 (missing closing }) : 20
Pattern 3 (extra text after name) : 1


In [33]:
def fix_early_closing_brace(tex_text):
    """Fix \mono{X}-rest} where } closed too early after stripping \\sfi"""
    
    # pattern: \mono{short}-text} {amount}{unit}
    # the } after short word is wrong closing, real } comes later
    tex_text = re.sub(
        r'(\\mono\{)([^}]{1,10})\}-([^}]+\})',  # \mono{X}-rest}
        r'\1\2-\3',                               # \mono{X-rest}
        tex_text
    )
    return tex_text

# test
test_cases = [
    r'\mono{N}-Acetyl-D-glucosamine} {1.0}{g}',
    r'\mono{p}-Aminobenzoic acid} {50.0}{mg}',
    r'\mono{myo}-Inositol} {5.0}{mg}',
]
for t in test_cases:
    print(f"BEFORE: {t}")
    print(f"AFTER : {fix_early_closing_brace(t)}")
    print()

BEFORE: \mono{N}-Acetyl-D-glucosamine} {1.0}{g}
AFTER : \mono{N-Acetyl-D-glucosamine} {1.0}{g}

BEFORE: \mono{p}-Aminobenzoic acid} {50.0}{mg}
AFTER : \mono{p-Aminobenzoic acid} {50.0}{mg}

BEFORE: \mono{myo}-Inositol} {5.0}{mg}
AFTER : \mono{myo-Inositol} {5.0}{mg}



<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_6852/2232726463.py:2: SyntaxWarning: invalid escape sequence '\m'
  """Fix \mono{X}-rest} where } closed too early after stripping \\sfi"""


In [35]:
# apply pattern 1 fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_early_closing_brace)

# now fix pattern 2 — missing closing } on name
def fix_missing_name_brace(tex_text):
    r"""Fix \mono{name with no closing } before amount"""
    
    # pattern: \mono{name followed by spaces then {amount}
    # no } before the {amount}
    tex_text = re.sub(
        r'(\\mono\{)([^}]+?)\s{2,}(\{[0-9])',  # \mono{name   {amount
        r'\1\2} \3',                             # \mono{name} {amount
        tex_text
    )
    return tex_text

# test on known cases
test_cases = [
    r'\mono{MgSO$_4$·7H$_2$O                                  {0.02}{g}',
    r'\mono{MnCl$_2$·4H$_2$O                                  {1.81}{g}',
]
for t in test_cases:
    print(f"BEFORE: {t.strip()}")
    print(f"AFTER : {fix_missing_name_brace(t).strip()}")
    print()

BEFORE: \mono{MgSO$_4$·7H$_2$O                                  {0.02}{g}
AFTER : \mono{MgSO$_4$·7H$_2$O} {0.02}{g}

BEFORE: \mono{MnCl$_2$·4H$_2$O                                  {1.81}{g}
AFTER : \mono{MnCl$_2$·4H$_2$O} {1.81}{g}



In [36]:
# apply pattern 2 fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_missing_name_brace)

# fix pattern 3 — extra text after name
def fix_extra_text_after_name(tex_text):
    r"""Fix \mono{name}extra_text} {amount}{unit}"""
    
    # pattern: \mono{name}word} — extra word before amount
    tex_text = re.sub(
        r'(\\mono\{[^}]+)\}([a-zA-Z]+)\}',  # \mono{name}word}
        r'\1 \2}',                            # \mono{name word}
        tex_text
    )
    return tex_text

# test on known case
test = r'\mono{1 M MgSO$_4$}solution} {1.0}{ml}'
print(f"BEFORE: {test}")
print(f"AFTER : {fix_extra_text_after_name(test)}")

BEFORE: \mono{1 M MgSO$_4$}solution} {1.0}{ml}
AFTER : \mono{1 M MgSO$_4$ solution} {1.0}{ml}


In [81]:
# apply pattern 3 fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_extra_text_after_name)

# recheck how many \mono lines still missing amount/unit
no_amount = []
for tex in df["tex_text_clean"]:
    lines = tex.splitlines()
    for line in lines:
        if '\\mono' in line and not re.search(unit_pattern, line):
            no_amount.append(line.strip())

print(f"\\mono lines still missing amount/unit: {len(no_amount)}")
print("\nRemaining examples:")
for line in no_amount[:10]:
    print(repr(line[:80]))

\mono lines still missing amount/unit: 89

Remaining examples:
'\\mono{MnSO$_4$·{x}H$_2$O} {1.0}{g}'
'\\mono{Concentrated {Vibrio} suspension (see below)} {10.0}{ml}'
'\\mono{MnSO$_4$·{x}H$_2$O} {4.5}{mg}'
'\\mono{NH$_4$Cl} {0.}5{g}'
'\\mono                                                         {0.2 M Cellobiose '
'\\mono{MnSO$_4$·{x}H$_2$O} {1.0}{g}'
'\\mono{Cr$_2$(SO$_4$)$_3$·{x}H$_2$O} {0.5}{g}'
'\\mono{p-}Aminobenzoic acid} {0.25}{mg}'
'\\mono{KCl} {0.34}'
'\\mono{MnSO$_4$·{x}H$_2$O} {500.0}{mg}'


In [37]:
p4 = [l for l in no_amount if re.search(r'\{[xX]\}', l)]
p5 = [l for l in no_amount if re.search(r'\{[A-Z][a-z]+\}', l)]
p6 = [l for l in no_amount if re.search(r'\{[0-9.]+\}[0-9]|\{[0-9.]+\}$', l)]
p7 = [l for l in no_amount if re.search(r'^\\mono\s+\{', l)]
p8 = [l for l in no_amount if re.search(r'\\mono\{[^}]+-\}', l)]

print(f"Pattern 4 ({{x}} in name - valid)     : {len(p4)}")
print(f"Pattern 5 ({{Genus}} in name - valid) : {len(p5)}")
print(f"Pattern 6 (corrupted amount)          : {len(p6)}")
print(f"Pattern 7 (broken \\mono tag)          : {len(p7)}")
print(f"Pattern 8 (early }} before hyphen)    : {len(p8)}")

Pattern 4 ({x} in name - valid)     : 75
Pattern 5 ({Genus} in name - valid) : 1
Pattern 6 (corrupted amount)          : 2
Pattern 7 (broken \mono tag)          : 1
Pattern 8 (early } before hyphen)    : 1


In [38]:
# print all non-valid patterns
fixable = [l for l in no_amount 
           if not re.search(r'\{[xX]\}', l)        # not pattern 4
           and not re.search(r'\{[A-Z][a-z]+\}', l)] # not pattern 5

print(f"Fixable remaining: {len(fixable)}")
for l in fixable:
    print(repr(l[:80]))

Fixable remaining: 107
'\\mono{N}-Acetyl-D-glucosamine} {1.0}{g}'
'\\mono{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{MgSO$_4$·7H$_2$O                                  {0.02}{g}'
'\\mono{1 M MgSO$_4$}solution} {1.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{MnCl$_2$·4H$_2$O                                  {1.81}{g}'
'\\mono{myo}-Inositol} {5.0}{mg}'
'\\mono{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {100.0}{mg}'
'\\mono{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{n}-Butyric acid} {0.4}{ml}'
'\\mono{iso}-Butyric acid} {0.4}{ml}'
'\\mono{n}-Valeric acid} {0.2}{ml}'
'\\mono{iso}-Valeric acid} {0.2}{ml}'
'\\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}'
'\\mono{NH$_4$Cl} {0.}5{g}'
'\\mono{8% NaHCO$_3$ solution*  {25.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {5.0}        {mg}'
'\\mono                                                         {0.2 M Cellobiose '
'\\mono{p-}Aminobenzoic acid} {0.25}{m

In [39]:
# fix group 1 — extra } 
def fix_extra_brace(tex_text):
    r"""Fix extra } in \mono lines"""
    tex_text = re.sub(r'\}\s*\}(\s*\{[0-9])', r'}\1', tex_text)  # } } {amount → } {amount
    tex_text = re.sub(r'\{([0-9.]+)\}\}', r'{\1}', tex_text)      # {amount}} → {amount}
    return tex_text

# test
test_cases = [
    r'\mono{15\% MgSO$_4$·7H$_2$O solution} } {20.0}{ml}',
    r'\mono{H$_3$BO$_4$} {0.02}}{g}',
    r'\mono{CaCl$_2$·2H$_2$O} {0.25}}{g}',
]
for t in test_cases:
    print(f"BEFORE: {t}")
    print(f"AFTER : {fix_extra_brace(t)}")
    print()

BEFORE: \mono{15\% MgSO$_4$·7H$_2$O solution} } {20.0}{ml}
AFTER : \mono{15\% MgSO$_4$·7H$_2$O solution} {20.0}{ml}

BEFORE: \mono{H$_3$BO$_4$} {0.02}}{g}
AFTER : \mono{H$_3$BO$_4$} {0.02}{g}

BEFORE: \mono{CaCl$_2$·2H$_2$O} {0.25}}{g}
AFTER : \mono{CaCl$_2$·2H$_2$O} {0.25}{g}



In [40]:
# apply group 1 fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_extra_brace)

# fix group 2 — corrupted amount {0.}5{g} → {0.5}{g}
def fix_corrupted_amount(tex_text):
    r"""Fix corrupted amounts like {0.}5{g}"""
    # {0.}5{g} → {0.5}{g}
    tex_text = re.sub(
        r'\{([0-9]+)\.\}([0-9]+)\{([a-zA-Zμ]+)\}',
        r'{\1.\2}{\3}',
        tex_text
    )
    # {0.34} missing unit → flag with placeholder
    tex_text = re.sub(
        r'(\\mono\{[^}]+\})\s*\{([0-9.]+)\}$',
        r'\1 {\2} {?}',
        tex_text
    )
    return tex_text

# fix group 4 — ] instead of }
def fix_wrong_bracket(tex_text):
    r"""Fix ] instead of } on unit"""
    tex_text = re.sub(r'\{([a-zA-Zμ]+)\]', r'{\1}', tex_text)
    return tex_text

# fix group 5 — early } before hyphen
def fix_early_brace_hyphen(tex_text):
    r"""Fix \mono{p-}Aminobenzoic → \mono{p-Aminobenzoic"""
    tex_text = re.sub(
        r'(\\mono\{[^}]{1,5}-)\}([^}]+\})',
        r'\1\2',
        tex_text
    )
    return tex_text

# test all
tests = {
    'corrupted amount': r'\mono{NH$_4$Cl} {0.}5{g}',
    'wrong bracket'   : r'\mono{1\% Sodium ascorbate solution} {10.0}{ml]',
    'early } hyphen'  : r'\mono{p-}Aminobenzoic acid} {0.25}{mg}',
}
for name, t in tests.items():
    print(f"\n{name}:")
    print(f"BEFORE: {t}")
    result = fix_corrupted_amount(fix_wrong_bracket(fix_early_brace_hyphen(t)))
    print(f"AFTER : {result}")


corrupted amount:
BEFORE: \mono{NH$_4$Cl} {0.}5{g}
AFTER : \mono{NH$_4$Cl} {0.5}{g}

wrong bracket:
BEFORE: \mono{1\% Sodium ascorbate solution} {10.0}{ml]
AFTER : \mono{1\% Sodium ascorbate solution} {10.0}{ml}

early } hyphen:
BEFORE: \mono{p-}Aminobenzoic acid} {0.25}{mg}
AFTER : \mono{p-Aminobenzoic acid} {0.25}{mg}


In [87]:
# apply all fixes
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_corrupted_amount)
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_wrong_bracket)
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_early_brace_hyphen)

# fix group 3 — missing closing } on name (MnCl case)
def fix_missing_name_brace2(tex_text):
    r"""Fix \mono{name (extra info) {amount} missing closing }"""
    tex_text = re.sub(
        r'(\\mono\{[^}]+\([^)]+\))\s*(\{[0-9])',
        r'\1} \2',
        tex_text
    )
    return tex_text

# fix group 6 — broken \mono with name on next line
def fix_broken_mono(tex_text):
    r"""Fix \mono with no name, name floating on next line"""
    tex_text = re.sub(
        r'\\mono\s+\n\s*(\{[^}]+\})',
        r'\\mono\1',
        tex_text
    )
    return tex_text

df["tex_text_clean"] = df["tex_text_clean"].apply(fix_missing_name_brace2)
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_broken_mono)

# recheck
no_amount = []
for tex in df["tex_text_clean"]:
    for line in tex.splitlines():
        if '\\mono' in line and not re.search(unit_pattern, line):
            no_amount.append(line.strip())

print(f"\\mono lines still missing amount/unit: {len(no_amount)}")
print("\nRemaining:")
for l in no_amount[:10]:
    print(repr(l[:80]))

\mono lines still missing amount/unit: 81

Remaining:
'\\mono{MnSO$_4$·{x}H$_2$O} {1.0}{g}'
'\\mono{Concentrated {Vibrio} suspension (see below)} {10.0}{ml}'
'\\mono{MnSO$_4$·{x}H$_2$O} {4.5}{mg}'
'\\mono                                                         {0.2 M Cellobiose '
'\\mono{MnSO$_4$·{x}H$_2$O} {1.0}{g}'
'\\mono{Cr$_2$(SO$_4$)$_3$·{x}H$_2$O} {0.5}{g}'
'\\mono{KCl} {0.34}'
'\\mono{MnSO$_4$·{x}H$_2$O} {500.0}{mg}'
'\\mono{MnSO$_4$·{x}H$_2$O} {10.0}{mg}'
'\\mono{MnSO$_4$·{x}H$_2$O} {0.5}{g}'


In [41]:
valid_patterns = [
    r'\{[xX]\}',           # {x} chemistry
    r'\{[A-Z][a-z]+\}',   # {Genus}
    r'\$\^\{[^}]+\}\$',   # $^{2+}$ chemistry
]

truly_invalid = []
for l in no_amount:
    is_valid = any(re.search(p, l) for p in valid_patterns)
    if not is_valid:
        truly_invalid.append(l)

print(f"Valid (leave for parser) : {len(no_amount) - len(truly_invalid)}")
print(f"Truly invalid            : {len(truly_invalid)}")
print("\nTruly invalid lines:")
for l in truly_invalid:
    print(repr(l[:80]))

Valid (leave for parser) : 79
Truly invalid            : 104

Truly invalid lines:
'\\mono{N}-Acetyl-D-glucosamine} {1.0}{g}'
'\\mono{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{MgSO$_4$·7H$_2$O                                  {0.02}{g}'
'\\mono{1 M MgSO$_4$}solution} {1.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{MnCl$_2$·4H$_2$O                                  {1.81}{g}'
'\\mono{myo}-Inositol} {5.0}{mg}'
'\\mono{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {100.0}{mg}'
'\\mono{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{n}-Butyric acid} {0.4}{ml}'
'\\mono{iso}-Butyric acid} {0.4}{ml}'
'\\mono{n}-Valeric acid} {0.2}{ml}'
'\\mono{iso}-Valeric acid} {0.2}{ml}'
'\\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}'
'\\mono{NH$_4$Cl} {0.}5{g}'
'\\mono{8% NaHCO$_3$ solution*  {25.0}{ml}'
'\\mono{p}-Aminobenzoic acid} {5.0}        {mg}'
'\\mono                                                       

In [42]:
# find medium with broken \mono
for idx, row in df.iterrows():
    if re.search(r'\\mono\s{5,}\{', row["tex_text_clean"]):
        print(f"\n--- {row['grmd']} ---")
        for line in row["tex_text_clean"].splitlines():
            if 'Cellobiose' in line or re.search(r'\\mono\s{5,}', line):
                print(repr(line.strip()))

# find medium with missing unit
for idx, row in df.iterrows():
    if re.search(r'\\mono\{KCl\}\s*\{0\.34\}$', row["tex_text_clean"], re.MULTILINE):
        print(f"\n--- {row['grmd']} ---")
        lines = row["tex_text_clean"].splitlines()
        for i, line in enumerate(lines):
            if 'KCl' in line:
                # print surrounding context
                start = max(0, i-1)
                end = min(len(lines), i+3)
                for l in lines[start:end]:
                    print(repr(l.strip()))


--- 1181 ---
'\\mono                                                         {0.2 M Cellobiose solution} {30.0}{ml}'

--- 1144 ---
'\\mono{NH$_4$Cl} {0.5}{g}'
'\\mono{KCl} {0.34}'
'\\mono{CaCl$_2$·2H$_2$O} {0.14}{g}'
'\\mono{KH$_2$PO$_4$} {0.14}{g}'


In [43]:
def fix_last_two(tex_text):
    r"""Fix broken \mono tag and missing unit"""
    
    # fix \mono spaces {name} → \mono{name}
    tex_text = re.sub(
        r'\\mono\s+(\{[^}]+\}\s*\{[0-9])',
        r'\\mono\1',
        tex_text
    )
    
    # fix missing unit — {0.34} at end of line → {0.34}{g}
    tex_text = re.sub(
        r'(\\mono\{KCl\}\s*\{[0-9.]+\})$',
        r'\1{g}',
        tex_text,
        flags=re.MULTILINE
    )
    
    return tex_text

# test on both mediums
for grmd in [1181, 1144]:
    test = df[df["grmd"] == grmd]["tex_text_clean"].values[0]
    fixed = fix_last_two(test)
    for line in fixed.splitlines():
        if 'Cellobiose' in line or 'KCl' in line:
            print(f"{grmd} → {repr(line.strip())}")

1181 → '\\mono{0.2 M Cellobiose solution} {30.0}{ml}'
1144 → '\\mono{KCl} {0.34}{g}'


In [44]:
# apply fix
df["tex_text_clean"] = df["tex_text_clean"].apply(fix_last_two)

# final recheck
no_amount = []
for tex in df["tex_text_clean"]:
    for line in tex.splitlines():
        if '\\mono' in line and not re.search(unit_pattern, line):
            no_amount.append(line.strip())

# separate valid from invalid
valid_patterns = [
    r'\{[xX]\}',
    r'\{[A-Z][a-z]+\}',
    r'\$\^\{[^}]+\}\$',
]

truly_invalid = [l for l in no_amount 
                 if not any(re.search(p, l) for p in valid_patterns)]

print(f"Total remaining        : {len(no_amount)}")
print(f"Valid (leave for parser): {len(no_amount) - len(truly_invalid)}")
print(f"Truly invalid          : {len(truly_invalid)}")

if truly_invalid:
    print("\nTruly invalid:")
    for l in truly_invalid:
        print(repr(l[:80]))

Total remaining        : 84
Valid (leave for parser): 79
Truly invalid          : 5

Truly invalid:
'\\mono{1 M MgSO$_4$}solution} {1.0}{ml}'
'\\mono{NH$_4$Cl} {0.}5{g}'
'\\mono{p-}Aminobenzoic acid} {0.25}{mg}'
'\\mono{MnCl$_2$·4H$_2$O (5.0 g/L in 0.01 N H$_2$SO$_4$) {100.0}{μl}'
'\\mono{1\\% Sodium ascorbate solution (filter--sterilized)} {10.0}{ml]'


In [45]:
print("=== FINAL VERIFICATION ===\n")

# 1. tags
print("--- Tags ---")
print("\\sfi    :", df["tex_text_clean"].str.contains(r'\\sfi').sum())
print("\\hspace :", df["tex_text_clean"].str.contains(r'\\hspace').sum())
print("\\mu     :", df["tex_text_clean"].str.contains(r'\\mu').sum())
print("\\cdot   :", df["tex_text_clean"].str.contains(r'\\cdot').sum())

# 2. medium types
print("\n--- Medium types ---")
standard = df[df["tex_text_clean"].str.strip().str.startswith(r'\mono')]
chu_based = df[df["tex_text_clean"].str.strip().str.startswith(r'\chu')]
print(f"standard : {len(standard)}")
print(f"chu based: {len(chu_based)}")
print(f"total    : {len(standard) + len(chu_based)}")

=== FINAL VERIFICATION ===

--- Tags ---
\sfi    : 0
\hspace : 0
\mu     : 0
\cdot   : 0

--- Medium types ---
standard : 954
chu based: 211
total    : 1165


In [93]:
# cross references
print("--- Cross references ---")
print("see Medium No.:", df["tex_text_clean"].str.contains(r'see Medium No').sum())
print("see below     :", df["tex_text_clean"].str.contains(r'see below').sum())

# visual check 1205
print("\n--- Medium 1205 cleaned ---")
print(df[df["grmd"] == 1205]["tex_text_clean"].values[0][:500])

# visual check 1326
print("\n--- Medium 1326 cleaned ---")
print(df[df["grmd"] == 1326]["tex_text_clean"].values[0][:500])

--- Cross references ---
see Medium No.: 556
see below     : 270

--- Medium 1205 cleaned ---
\mono{NH$_4$Cl} {0.3}{g}
\mono{MgSO$_4$·7H$_2$O} {0.1}{g}
\mono{Modified Wolfe's mineral solution (see Medium No. [915])} {5.0}{ml}
\mono{Casamino acids (BD-Difco)} {0.1}{g}
\mono{Yeast extract} {0.02}{g}
\mono{Resazurin} {0.5}{mg}
\mono{Distilled water} {957.0}{ml}
\chu{Mix components thoroughly, adjust pH to 7.0 and autoclave under a N$_2$ gas atmosphere.  After cooling, aseptically and anaerobically add the following solutions from anaerobic stocks (autoclaved or *filter-sterilized):}

\mono{

--- Medium 1326 cleaned ---
\chu{Solution A:}
\mono{K$_2$HPO$_4$} {0.5}{g}
\mono{KH$_2$PO$_4$} {0.5}{g}
\mono{NH$_4$CI} {1.0}{g}
\mono{Na$_2$SO$_4$} {1.0}{g}
\mono{MgSO$_4$·7H$_2$O} {2.0}{g}
\mono{CaCl$_2$·2H$_2$O} {0.1}{g}
\mono{Sodium lactate} {2.0}{g}
\mono{Yeast extract} {1.0}{g}
\mono{FeCl$_2$ solution (see Medium No. [187])} {1.0}{ml}
\mono{Trace element solution (see Medium No. [187])} {1.0}{ml

In [94]:
df[["grmd", "md_name", "tex_text_clean"]].to_csv(
    '/home/sheikh/Projects/Thesis/data/medium_clean.csv',
    index=False
)
print(f"Saved successfully!")
print(f"Rows : {len(df)}")
print(f"Columns: {['grmd', 'md_name', 'tex_text_clean']}")

Saved successfully!
Rows : 1165
Columns: ['grmd', 'md_name', 'tex_text_clean']
